# Impala67 AI – QLoRA Finetuning (Google Colab, Flache Tools)
Trainiert **Qwen 3.5 4B** (am besten auf A100/H100, sonst L4/T4) mit Unsloth auf den echten flachen Tool-Trajektorien (`tool_defs_flat.json`). LoRA auf Attention, Gated-DeltaNet und MLP.

**Datenquelle (automatisch):**
Entweder `impala67_colab_bundle.zip` (oder die 4 Einzeldateien `chat_format.py`, `tool_defs_flat.json`, `impala_flat_train.jsonl`, `impala_flat_val.jsonl`) direkt nach `/content` hochladen – oder als Fallback aus `Google Drive/Impala67-Training/`.

## 0. Trainingsdaten & Skripte prüfen (Bundle/Upload oder Google Drive)

In [ ]:
import os, shutil, zipfile

os.chdir("/content")
NEEDED = ["chat_format.py", "tool_defs_flat.json", "impala_flat_train.jsonl", "impala_flat_val.jsonl"]

if os.path.exists("impala67_colab_bundle.zip"):
    with zipfile.ZipFile("impala67_colab_bundle.zip", "r") as zf:
        zf.extractall("/content")
    print("Bundle entpackt.")

missing = [f for f in NEEDED if not os.path.exists(f)]
if missing or os.path.exists("/content/drive/MyDrive/Impala67-Training") or not os.path.exists("impala67_colab_bundle.zip"):
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = "/content/drive/MyDrive/Impala67-Training"
    for f in NEEDED:
        src_f = f"{SRC}/{f}"
        assert os.path.exists(src_f), f"Fehlt in /content und {SRC}: {f}"
        shutil.copy(src_f, f)

for f in NEEDED:
    print(f"OK: {f} ({os.path.getsize(f)} Bytes)")

## 1. Unsloth installieren

In [ ]:
%%capture
!pip install unsloth

## 2. Modell laden (4-Bit) und LoRA-Adapter anlegen

In [ ]:
from unsloth import FastLanguageModel
import torch

assert torch.cuda.is_available(), "Keine GPU aktiv – bitte Laufzeittyp auf T4/L4/A100 GPU stellen!"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
gpu_name = torch.cuda.get_device_name(0)
VERSION = "v4"
max_seq_length = 16384 if vram_gb >= 20 else 8192
model_name = "unsloth/Qwen3.5-4B"
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB) -> Modell: {model_name}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Das Chat-Template des Tokenizers bleibt unverändert (Qwen-eigenes Tool- und Thinking-Format).
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",            # 8 Full-Attention-Schichten
                      "in_proj_qkv", "in_proj_z", "out_proj",         # 24 Gated-DeltaNet-Schichten
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
model.print_trainable_parameters()

## 3. Daten mit dem Qwen-Template (flache Werkzeuge) rendern

In [ ]:
from datasets import Dataset
from chat_format import INSTRUCTION_PART, RESPONSE_PART, build_texts, load_tools

tools = load_tools("tool_defs_flat.json")
train_texts = build_texts(tokenizer, "impala_flat_train.jsonl", tools, max_seq_length)
val_texts = build_texts(tokenizer, "impala_flat_val.jsonl", tools, max_seq_length)
train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts}) if val_texts else None
print(f"Tools: {len(tools)} | Training: {len(train_ds)} | Validierung: {len(val_texts)}")

print(train_ds[0]["text"][-1500:])

## 4. Training (3 Epochen, Loss nur auf Assistenten-Antworten)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

bf16 = torch.cuda.is_bf16_supported()  # T4: False -> fp16
has_val = val_ds is not None and len(val_ds) > 0

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        packing = False,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 3,
        warmup_ratio = 0.05,
        learning_rate = 1e-4,
        bf16 = bf16,
        fp16 = not bf16,
        logging_steps = 1,
        eval_strategy = "epoch" if has_val else "no",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        save_strategy = "no",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(trainer, instruction_part = INSTRUCTION_PART, response_part = RESPONSE_PART)
trainer_stats = trainer.train()
print(trainer_stats)

## 5. Kurztest vor dem Export
Muss einen `<tool_call>` mit `<function=edit_page>` und `edits` (oder erst `read_pages`) erzeugen.

In [ ]:
FastLanguageModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
msgs = [
    {"role": "system", "content": "Du bist der KI-Coach von Impala67, einer lokalen Notiz- und Lern-App. Antworte auf Deutsch, kompakt. Formeln als LaTeX ($...$ inline, $$...$$ als Block).\nDu kannst die App direkt bedienen. Lies erst nach (search, dann read_pages), dann ändere so klein wie möglich (edit_page mit edits, append oder props statt content)."},
    {"role": "user", "content": "Auf meiner bereits gelesenen Seite 'Physik 1' steht aktuell 'v = s * t'. Korrigiere genau diese Stelle per edit_page zu '$v = s / t$'."},
]
prompt = tok.apply_chat_template(msgs, tools = tools, tokenize = False, add_generation_prompt = True)
inputs = tok(prompt, return_tensors = "pt").to("cuda")
out = model.generate(**inputs, max_new_tokens = 400, do_sample = False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:]))

## 6. LoRA-Adapter in Google Drive sichern

In [ ]:
import os, shutil

# LoRA-Adapter zuerst sichern: Der GGUF-Export lässt sich so ohne neues Training wiederholen.
model.save_pretrained("impala67_lora")
tokenizer.save_pretrained("impala67_lora")
OUT_DRIVE = "/content/drive/MyDrive/Impala67-Training/output"
if os.path.exists("/content/drive/MyDrive"):
    dst = os.path.join(OUT_DRIVE, f"impala67_lora_{VERSION}")
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree("impala67_lora", dst)
    print(f"Adapter in Google Drive gespeichert: {dst}")
else:
    print("Kein Google Drive eingebunden – Adapter liegt nur in /content/impala67_lora")

## 7. GGUF exportieren (Q4_K_M) nach Google Drive

In [ ]:
import glob, os, shutil

model.save_pretrained_gguf("impala67_model", tokenizer, quantization_method = "q4_k_m")

OUT_DRIVE = "/content/drive/MyDrive/Impala67-Training/output"
ggufs = sorted(set(glob.glob("impala67_model*/*.gguf") + glob.glob("*.gguf")))
print("GGUF-Dateien:", ggufs)
for f in ggufs:
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(OUT_DRIVE, exist_ok=True)
        dst = os.path.join(OUT_DRIVE, f"impala67-{VERSION}-" + os.path.basename(f))
        shutil.copy(f, dst)
        print(f"In Google Drive gespeichert: {dst}")
    else:
        from google.colab import files
        files.download(f)
if os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.flush_and_unmount()  # wartet, bis alles hochgeladen ist
    print("Google-Drive-Upload abgeschlossen.")

## 8. GGUF-Schnelltest mit llama.cpp
Lädt das exportierte GGUF mit dem von Unsloth gebauten `llama-server` und prüft zwei Prompts. v2 kippte genau hier in Wiederholschleifen („单单单…“) – dann ist der Export kaputt, nicht das Training.

In [ ]:
import glob, json, os, re, subprocess, time, urllib.request

def find_bin(name):
    hits = [p for p in glob.glob(f"/root/.unsloth/**/{name}", recursive=True) + glob.glob(f"/content/**/{name}", recursive=True) if os.access(p, os.X_OK)]
    return hits[0] if hits else None

main_gguf = [g for g in ggufs if "mmproj" not in g.lower()][0]
server = find_bin("llama-server")
assert server, "llama-server nicht gefunden – Unsloth hat llama.cpp ohne Server gebaut"
proc = subprocess.Popen([server, "-m", main_gguf, "-c", "8192", "-ngl", "99", "--jinja", "--port", "8099"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8099/health", timeout=2); break
    except Exception:
        time.sleep(2)

def ask(messages, tools=None, max_tokens=1024):
    body = {"messages": messages, "max_tokens": max_tokens, "temperature": 0.6, "top_p": 0.95, "top_k": 20}
    if tools: body["tools"] = [{"type": "function", "function": t} if "function" not in t else t for t in tools]
    req = urllib.request.Request("http://127.0.0.1:8099/v1/chat/completions", json.dumps(body).encode(), {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]

def looped(text):
    return bool(re.search(r"(.{1,8})\1{15,}", text or ""))

ok = True
r1 = ask([{"role": "user", "content": "Sag nur Hallo in einem Wort."}])
t1 = (r1["message"].get("reasoning_content") or "") + (r1["message"].get("content") or "")
print("Test 1:", r1["finish_reason"], repr((r1["message"].get("content") or "")[:200]))
ok &= r1["finish_reason"] == "stop" and not looped(t1)

r2 = ask(msgs, tools=tools)
calls = r2["message"].get("tool_calls") or []
print("Test 2:", r2["finish_reason"], [c["function"]["name"] for c in calls])
ok &= bool(calls) and not looped(json.dumps(r2["message"], ensure_ascii=False))

proc.terminate()
print("GGUF-SCHNELLTEST:", "BESTANDEN" if ok else "FEHLGESCHLAGEN")